In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

# Define paths to the data files
data_folder = '/content/drive/MyDrive/datasets/manipuri MT/English-Manipuri/parallel'
train_src_file = os.path.join(data_folder, 'en-mni-train-mni.txt')
train_tgt_file = os.path.join(data_folder, 'en-mni-train-en.txt')
valid_src_file = os.path.join(data_folder, 'en-mni-valid-mni.txt')
valid_tgt_file = os.path.join(data_folder, 'en-mni-valid-en.txt')
test_src_file = os.path.join(data_folder, 'en-mni-test-mni.txt')
test_tgt_file = os.path.join(data_folder, 'en-mni-test-en.txt')

def load_sentences(src_file, tgt_file):
    src_sentences = []
    tgt_sentences = []
    with open(src_file, 'r', encoding='utf-8') as src_f, open(tgt_file, 'r', encoding='utf-8') as tgt_f:
        for src_line, tgt_line in zip(src_f, tgt_f):
            src_sentences.append(src_line.strip())
            tgt_sentences.append(tgt_line.strip())
    return src_sentences, tgt_sentences

train_src, train_tgt = load_sentences(train_src_file, train_tgt_file)
valid_src, valid_tgt = load_sentences(valid_src_file, valid_tgt_file)
test_src, test_tgt = load_sentences(test_src_file, test_tgt_file)

print("Sample Training Source Sentences:", train_src[:3])
print("Sample Training Target Sentences:", train_tgt[:3])

Sample Training Source Sentences: ['এনর্জি পোর্টেল অসিনা মথক্কী ৱারোলশীং অসিগী ঈ-পাউ অসি নহাক্না মখোয়বু শীজিন্ননিংগদবা অমদি মখোয়গা লোয়নরিবা কান্নবশীং লৌবা পামহন্নবগী মওংদা পুক্নিং থৌগৎনিংঙাই ওইবা ৱারিশীংগা লোয়ননা পীনবা হোৎনৈ ।', 'মহৌশাগী উপদ্রবশীংনা শোকহল্লবা লৌমীশীংদা রিলিফ পীনবগীদমক্তা ,  বেঙ্কশীংদা অহানবা চহিদুগী ওইনা রিষ্ট্রকচর তৌরবা এমাউন্ট অদুদা ২%গী ইন্টরেষ্ট সবভেন্সন পীগনি ।', 'প্রধানমন্ত্রী শ্রী নরেন্দ্র মোদীনা লুচিংবা কেন্দ্রগী মন্ত্রীমন্দলনা দিপার্তমেন্ত ওফ ইকোনোমীক এফিয়ার্স ( ইন্দিয়ান ইকোনোমীক সর্বিস কেদর )  অমসুং দি ত্রেজরী , গবর্নমেন্ত ওফ ওস্ত্রেলিয়াগী মরক্তা থা অহুমগী ওইনা সেকেন্দমেন্ত প্রোগ্রামগীদমক মেমোরেন্দা ওফ অন্দর্সতেন্দিং খুৎয়েক পিন্নবা য়াখ্রে ।']
Sample Training Target Sentences: ['the energy portal attempts to give information on the above aspects with inspiring stories that would motivate you to use them and derive the associated benefits .', 'to provide relief to the farmers affected by natural calamities , the interest subvention of 2 % will be provided to ba

In [4]:
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn.utils.rnn as rnn_utils

def tokenize(sentence):
    return sentence.split()

def build_vocab(sentences):
    vocab = set()
    for sentence in sentences:
        vocab.update(tokenize(sentence))
    return {word: idx for idx, word in enumerate(vocab, start=2)}  # Start from 2 to reserve 0 and 1 for <pad> and <unk>

manipuri_vocab = build_vocab(train_src)
english_vocab = build_vocab(train_tgt)

# Add special tokens
manipuri_vocab['<pad>'] = 0
manipuri_vocab['<unk>'] = 1
english_vocab['<pad>'] = 0
english_vocab['<unk>'] = 1

class TranslationDataset(Dataset):
    def __init__(self, source_sentences, target_sentences, source_vocab, target_vocab):
        self.source_sentences = source_sentences
        self.target_sentences = target_sentences
        self.source_vocab = source_vocab
        self.target_vocab = target_vocab

    def __len__(self):
        return len(self.source_sentences)

    def __getitem__(self, idx):
        source_sentence = self.source_sentences[idx]
        target_sentence = self.target_sentences[idx]
        source_indices = [self.source_vocab.get(word, self.source_vocab['<unk>']) for word in tokenize(source_sentence)]
        target_indices = [self.target_vocab.get(word, self.target_vocab['<unk>']) for word in tokenize(target_sentence)]
        return torch.tensor(source_indices), torch.tensor(target_indices)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = rnn_utils.pad_sequence(src_batch, padding_value=manipuri_vocab['<pad>'])
    tgt_batch = rnn_utils.pad_sequence(tgt_batch, padding_value=english_vocab['<pad>'])
    return src_batch, tgt_batch

train_dataset = TranslationDataset(train_src, train_tgt, manipuri_vocab, english_vocab)
valid_dataset = TranslationDataset(valid_src, valid_tgt, manipuri_vocab, english_vocab)
test_dataset = TranslationDataset(test_src, test_tgt, manipuri_vocab, english_vocab)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
valid_dataloader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

# Inspect some samples from the training, validation, and test datasets
for i, (src, tgt) in enumerate(train_dataloader):
    print("Training Sample Source:", src[:, 0].tolist())
    print("Training Sample Target:", tgt[:, 0].tolist())
    break

for i, (src, tgt) in enumerate(valid_dataloader):
    print("Validation Sample Source:", src[:, 0].tolist())
    print("Validation Sample Target:", tgt[:, 0].tolist())
    break

for i, (src, tgt) in enumerate(test_dataloader):
    print("Test Sample Source:", src[:, 0].tolist())
    print("Test Sample Target:", tgt[:, 0].tolist())
    break

Training Sample Source: [20048, 33997, 33201, 14451, 6391, 42539, 79, 10628, 32095, 41807, 18376, 1849, 2043, 39998, 34933, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Training Sample Target: [15929, 20768, 2, 9178, 19953, 14426, 18520, 5574, 6282, 8364, 8563, 5162, 7158, 5084, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Validation Sample Source: [20063, 1874, 23288, 20774, 19091, 35250, 20063, 1874, 1846, 33342, 25234, 30131, 14088, 11408, 21950, 35421, 14903, 34933, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Validation Sample Target: [14724, 18018, 1809, 8488, 17594, 7300, 868, 8488, 5216, 12597, 20873, 10534, 3004, 3638, 21908, 12442, 1376, 17035, 22484, 10128, 8488, 21281, 9438, 635, 5084, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [5]:
#Seq2Seq model architecture
import torch.nn as nn
import random
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len = trg.shape[0]
        batch_size = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        input = trg[0,:]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            top1 = output.argmax(1)
            input = trg[t] if random.random() < teacher_forcing_ratio else top1
        return outputs

INPUT_DIM = len(manipuri_vocab)
OUTPUT_DIM = len(english_vocab)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

In [ ]:
!pip install torchviz

  Preparing metadata (setup.py) ... done
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl (176.2 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (99 kB)
     ━━━━━━━

In [ ]:
import torch.optim as optim
from tqdm import tqdm

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=english_vocab['<pad>'])

def train(model, dataloader, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0

    for i, (src, trg) in enumerate(tqdm(dataloader, desc="Epoch Training")):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg)
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(dataloader)

N_EPOCHS = 10
CLIP = 1

for epoch in range(N_EPOCHS):
    print(f'Epoch: {epoch+1}')
    train_loss = train(model, train_dataloader, optimizer, criterion, CLIP)
    print(f'Train Loss: {train_loss:.4f}')

NameError: name 'model' is not defined

In [ ]:
def evaluate(model, dataloader, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for i, (src, trg) in enumerate(dataloader):
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, 0)
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(dataloader)

eval_loss = evaluate(model, valid_dataloader, criterion)
print(f'Evaluation Loss: {eval_loss:.4f}')


NameError: name 'model' is not defined

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/model.pt")


In [ ]:
test_loss = evaluate(model, test_dataloader, criterion)
print(f'Test Loss: {test_loss:.4f}')

Test Loss: 7.0229


In [ ]:
test_loss = evaluate(model, test_dataloader, criterion)
print(f'Test Loss: {test_loss:.4f}')


Test Loss: 7.0229


In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [32]:
# prompt: test the model by using random sentences from the test data and then compare with original test target data

import random

# Choose a random sample from the test set
random_idx = random.randint(0, len(test_dataset))
src_sentence = test_dataset.source_sentences[random_idx]
tgt_sentence = test_dataset.target_sentences[random_idx]

# Convert the source sentence to a tensor and move it to the device
src_tensor = torch.tensor(
    [manipuri_vocab.get(word, manipuri_vocab['<unk>']) for word in tokenize(src_sentence)]
).unsqueeze(0).to(device)

# Generate the translation
model.eval()
with torch.no_grad():
    output = model(src_tensor, torch.tensor([[english_vocab['<sos>']]]).to(device))
    #translated_sentence = [english_vocab[idx] for idx in output.argmax(2)[:, 0].tolist()]
    translated_sentence = [english_vocab.get(idx, '<unk>') for idx in output.argmax(2)[:, 0].tolist()]

# Print the source, target, and translated sentences
print("Source Sentence:", src_sentence)
print("Target Sentence:", tgt_sentence)
print("Translated Sentence:", " ".join(translated_sentence))


Source Sentence: চৎপা মতমদা চাং পুবদা ৱাবা
Target Sentence: difficulty keeping balance while walking
Translated Sentence: <unk>


In [33]:
# prompt: test the model by using  sentences from the test data and then compare with original test target data

# Test the model on multiple sentences from the test dataset
for i in range(5):
    random_idx = random.randint(0, len(test_dataset))
    src_sentence = test_dataset.source_sentences[random_idx]
    tgt_sentence = test_dataset.target_sentences[random_idx]

    # Convert the source sentence to a tensor and move it to the device
    src_tensor = torch.tensor(
        [manipuri_vocab.get(word, manipuri_vocab['<unk>']) for word in tokenize(src_sentence)]
    ).unsqueeze(0).to(device)

    # Generate the translation
    model.eval()
    with torch.no_grad():
        output = model(src_tensor, torch.tensor([[english_vocab['<sos>']]]).to(device))
        translated_sentence = [english_vocab.get(idx, '<unk>') for idx in output.argmax(2)[:, 0].tolist()]

    # Print the source, target, and translated sentences
    print("Source Sentence:", src_sentence)
    print("Target Sentence:", tgt_sentence)
    print("Translated Sentence:", " ".join(translated_sentence))
    print()


Source Sentence: মীওইশীংগী মায়োক্তা চত্থবা সাইবরক্রাইমগী মতাংদা অহেনবা ইনফোর্মেসন খরা পীবীয়ু ।
Target Sentence: tell us more information about cybercrimes against persons ?
Translated Sentence: <unk>

Source Sentence: ভারত সরকারগী শেন্থং মখাদা পায়খৎকদৌরিবা কারতারপুর কোরিদোর অসিনা মতমগা চুনরবা খুদোংচাবশিং পীদুনা ইরাই লাইনা নুংঙাইনা চৎপগী খুদোংচাবা ফংগনি।
Target Sentence: The Kartarpur corridor will be implemented as an integrated development project with Government of India funding, to provide smooth and easy passage, with all the modern amenities.
Translated Sentence: <unk>

Source Sentence: দিজিটেল পেমেন্টশীং প্রোমোট তৌনবগীদমক্তা দিজিশালা চেনেল হৌদোকখ্রে
Target Sentence: digishala channel launched to promote digital payments
Translated Sentence: <unk>

Source Sentence: জাতিগী লনশীং য়োক্তুনা  ,   ঙাক্তুনা থম্নবা অমসুং প্রোমোট তৌনবা অমসুং ইমপ্লোয়মেন্ট অদুদগী থোরকপা অশোক-অপন অমসুং লায়নাশীং মুত্থৎনবা জোইন্ট লেবর-মেনেজমেন্টগী থবক-থৌরমশীং পুক্নিং থৌগত্তুনা ;
Target Sentence: encourage joint

In [42]:
!pip install --upgrade sacrebleu

In [48]:
from sacrebleu.metrics import BLEU

# Load the BLEU metric
bleu = BLEU()

# Define a generate function (if not already defined)
def generate(model, dataloader, english_vocab):
    model.eval()
    generated_sentences = []
    with torch.no_grad():
        for src_tensor, _ in dataloader:
            src_tensor = src_tensor.to(device)
            output = model(src_tensor, torch.tensor([[english_vocab['<sos>']]]).to(device))
            translated_sentence = [english_vocab.get(idx, '<unk>') for idx in output.argmax(2)[:, 0].tolist()]
            generated_sentences.append(" ".join(translated_sentence))
    return generated_sentences

# Calculate the BLEU score between the target and translated sentences
generated_sentences = generate(model, test_dataloader, english_vocab) # Call the generate function
# Change the format of references to a list of lists of strings
references = [[test_dataset.target_sentences[i]] for i in range(len(test_dataset))]
bleu_score = bleu.corpus_score(generated_sentences, references)

# Print the BLEU score
print(f"BLEU Score: {bleu_score.score:.2f}")

BLEU Score: 0.00
